## 1. Necessary Imports

In [261]:
import pandas as pd
import numpy as np
import re
import json
import time
import warnings
import os



## 2. Data Cleaning & Enrichment

Steps performed:
- Parse mixed-format timestamps (UTC → IST)
- Decode the JSON-array `violation_type` column into clean categories
- Assign a **congestion impact weight** (1–5) per violation type
- Assign a **vehicle size weight** (1–5) per vehicle type
- Compute a composite `risk_score_raw` = impact × vehicle weight

In [262]:
# ── EDIT THIS PATH to match your Kaggle dataset ──────────────
DATA_PATH = "jan to may police violation_anonymized791b166.csv"

df = pd.read_csv(DATA_PATH)
print(f"✓ Loaded {len(df_raw):,} rows × {len(df_raw.columns)} columns\n")
df.head()


✓ Loaded 298,450 rows × 24 columns



,id,latitude,longitude,location,vehicle_number,vehicle_type,description,violation_type,offence_code,created_datetime,...,center_code,police_station,data_sent_to_scita,junction_name,action_taken_timestamp,data_sent_to_scita_timestamp,updated_vehicle_number,updated_vehicle_type,validation_status,validation_timestamp
0,FKID000000,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,NaN,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00,...,9.0,Madiwala,True,No Junction,NaN,NaN,FKN00GL0000,MAXI-CAB,approved,2023-11-30 03:08:24.818+00
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,NaN,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00,...,82.0,Bellandur,False,No Junction,NaN,NaN,NaN,NaN,NaN,NaN
2,FKID000002,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,NaN,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00,...,9.0,Madiwala,True,No Junction,NaN,NaN,FKN00GL0002,MAXI-CAB,approved,2023-11-30 03:08:56.998+00
3,FKID000003,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,NaN,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00,...,26.0,Byatarayanapura,True,No Junction,NaN,NaN,FKN00GL0003,SCOOTER,approved,2023-11-18 23:35:12.428+00
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,NaN,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00,...,3.0,Upparpet,True,BTP044 - Sagar Theatre Junction,NaN,NaN,FKN00GL0004,TANKER,approved,2023-11-30 03:11:32.796+00


In [263]:
# ── Parse datetime (handles mixed formats with/without microseconds) ──
df["created_datetime"] = pd.to_datetime(df["created_datetime"], format="mixed", utc=True)
df["created_ist"] = df["created_datetime"].dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
df["hour_ist"]    = df["created_ist"].dt.hour
df["date"]        = df["created_ist"].dt.date
df["day_of_week"] = df["created_ist"].dt.day_name()
df["month"]       = df["created_ist"].dt.to_period("M").astype(str)

print(f"✓ Datetime parsed | range: {df['date'].min()} → {df['date'].max()}")


✓ Datetime parsed | range: 2023-11-10 → 2024-04-08


In [264]:
df_raw['validation_status'].value_counts()

validation_status
approved      115400
rejected       49754
created1        7044
processing       678
duplicate        320
Name: count, dtype: int64

In [265]:
# --------------------------------------------------
# 1. APPROVED / CREATED
# --------------------------------------------------
accepted = df[
    df["validation_status"].isin(
        ["approved", "created1"]
    )
]

# --------------------------------------------------
# 2. DUPLICATES
# Keep if same vehicle/type/offence appears
# on a DIFFERENT DATE
# --------------------------------------------------
duplicates = df[df["validation_status"] == "duplicate"].copy()

non_duplicates = df[
    df["validation_status"] != "duplicate"
].copy()

keep_dup_idx = []

for idx, dup in duplicates.iterrows():

    matches = non_duplicates[
        (non_duplicates["vehicle_number"] == dup["vehicle_number"]) &
        (non_duplicates["vehicle_type"] == dup["vehicle_type"]) &
        (non_duplicates["violation_type"] == dup["violation_type"])
    ]

    if len(matches) == 0:
        continue

    # keep duplicate only if no match exists on same date
    if not (matches["date"] == dup["date"]).any():
        keep_dup_idx.append(idx)

duplicates_keep = duplicates.loc[keep_dup_idx]

# --------------------------------------------------
# FINAL DATAFRAME
# --------------------------------------------------
df = pd.concat(
    [
        accepted,
        duplicates_keep
    ],
    ignore_index=True
)

print(f"Final rows: {len(df):,}")

print("\nValidation status distribution:")
print(df["validation_status"].value_counts())

Final rows: 122,460

Validation status distribution:
validation_status
approved     115400
created1       7044
duplicate        16
Name: count, dtype: int64


In [266]:
df.columns

Index(['id', 'latitude', 'longitude', 'location', 'vehicle_number',
       'vehicle_type', 'description', 'violation_type', 'offence_code',
       'created_datetime', 'closed_datetime', 'modified_datetime', 'device_id',
       'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita',
       'junction_name', 'action_taken_timestamp',
       'data_sent_to_scita_timestamp', 'updated_vehicle_number',
       'updated_vehicle_type', 'validation_status', 'validation_timestamp',
       'created_ist', 'hour_ist', 'date', 'day_of_week', 'month'],
      dtype='object')

In [267]:
df.shape

(122460, 29)

In [269]:
import ast

# ── Congestion impact weight per violation type ──────────────
# Higher weight = greater impact on traffic flow (main road > footpath > etc.)
IMPACT_WEIGHTS = {
    "PARKING IN A MAIN ROAD": 5,
    "PARKING NEAR ROAD CROSSING": 4,
    "PARKING OTHER THAN BUS STOP": 2,
    "PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS": 4,
    "PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC": 4,
    "DOUBLE PARKING": 3,
    "NO PARKING": 3,
    "WRONG PARKING": 3,
    "PARKING OPPOSITE TO ANOTHER PARKED VEHICLE": 3,
    "PARKING ON FOOTPATH": 2,
    "H T V PROHIBITED": 3,
    "DEFECTIVE NUMBER PLATE": 1,
    "FAIL TO USE SAFETY BELTS": 1,
    "WITHOUT SIDE MIRROR": 1,
    "OBSTRUCTING DRIVER": 1,
    "DEMANDING EXCESS FARE": 1,
    "REFUSE TO GO FOR HIRE": 1
}

df['violation_type'] = df['violation_type'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

def get_primary_violation(violations):
    if not isinstance(violations, list) or len(violations) == 0:
        return None

    return max(
        violations,
        key=lambda x: IMPACT_WEIGHTS.get(x, 0)
    )

df['primary_violation'] = df['violation_type'].apply(get_primary_violation)

df["violation_impact"] = (
    df["primary_violation"]
    .map(IMPACT_WEIGHTS)
    .fillna(1)
    .astype(int)
)

# ── Vehicle size weight (bigger vehicle = more lane-width blocked) ───
VEHICLE_WEIGHTS = {
    # Two-wheelers
    "MOTOR CYCLE": 1,
    "SCOOTER": 1,
    "MOPED": 1,

    # Small passenger vehicles
    "PASSENGER AUTO": 2,
    "GOODS AUTO": 2,
    "CAR": 2,
    "JEEP": 2,

    # Medium vehicles
    "MAXI-CAB": 3,
    "VAN": 3,
    "TEMPO": 3,
    "SCHOOL VEHICLE": 3,
    "OTHERS": 3,

    # Large commercial vehicles
    "LGV": 4,
    "MINI LORRY": 4,
    "LORRY/GOODS VEHICLE": 4,
    "TRACTOR": 4,
    "TOURIST BUS": 4,

    # Heavy vehicles
    "HGV": 5,
    "PRIVATE BUS": 5,
    "BUS (BMTC/KSRTC)": 5,
    "FACTORY BUS": 5,
    "TANKER": 5,
    "TRAILER": 5
}

df["vehicle_weight"] = (
    df["updated_vehicle_type"]
    .str.upper()
    .map(VEHICLE_WEIGHTS)
    .fillna(1)
    .astype(int)
)

# ── Composite congestion risk score ───────────────────────────
df["risk_score_raw"] = (
    df["violation_impact"] *
    df["vehicle_weight"]
)

print(
    f"✓ Risk scoring complete | max risk_score_raw: "
    f"{df['risk_score_raw'].max()}"
)

df[
    [
        "primary_violation",
        "violation_impact",
        "vehicle_type",
        "vehicle_weight",
        "risk_score_raw"
    ]
].head()

✓ Risk scoring complete | max risk_score_raw: 25


,primary_violation,violation_impact,vehicle_type,vehicle_weight,risk_score_raw
0,PARKING NEAR ROAD CROSSING,4,CAR,3,12
1,PARKING IN A MAIN ROAD,5,CAR,3,15
2,NO PARKING,3,SCOOTER,1,3
3,NO PARKING,3,TANKER,5,15
4,NO PARKING,3,SCOOTER,1,3


## 3. H3 Hexagonal Spatial Indexing

We bin every violation into an **Uber H3 hex cell** (resolution 8 ≈ 460m per side).
This turns scattered lat/lon points into clean, comparable **patrol zones** —
each with an aggregated risk score, dominant violation type, and peak hour.

In [270]:
H3_RES = 8  # ~460m hex — good granularity for patrol zone sizing

df["h3_cell"] = df.apply(
    lambda r: h3.latlng_to_cell(r["latitude"], r["longitude"], H3_RES), axis=1
)

n_cells = df["h3_cell"].nunique()
print(f"✓ Assigned {n_cells} unique H3 cells at resolution {H3_RES}")


✓ Assigned 682 unique H3 cells at resolution 8


In [271]:
df.columns

Index(['id', 'latitude', 'longitude', 'location', 'vehicle_number',
       'vehicle_type', 'description', 'violation_type', 'offence_code',
       'created_datetime', 'closed_datetime', 'modified_datetime', 'device_id',
       'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita',
       'junction_name', 'action_taken_timestamp',
       'data_sent_to_scita_timestamp', 'updated_vehicle_number',
       'updated_vehicle_type', 'validation_status', 'validation_timestamp',
       'created_ist', 'hour_ist', 'date', 'day_of_week', 'month',
       'primary_violation', 'violation_impact', 'vehicle_weight',
       'risk_score_raw', 'h3_cell'],
      dtype='object')

In [272]:
daily_h3_risk = (
    df
    .groupby(['h3_cell', 'date'])
    .agg(
        total_risk=('risk_score_raw', 'sum'),
        total_chalans=('risk_score_raw', 'size')
    )
    .reset_index()
)

daily_h3_risk.head()

,h3_cell,date,total_risk,total_chalans
0,8860144a2dfffff,2023-11-29,48,4
1,8860144b4dfffff,2023-11-15,57,19
2,8860144b5bfffff,2023-11-29,30,3
3,8860144b5bfffff,2023-11-30,12,1
4,8860144b5bfffff,2023-12-01,15,1


In [273]:
daily_h3_risk.shape

(14403, 4)

In [274]:
daily_h3_risk.shape

(14403, 4)

## 4. Creating neighbour lag features

In [275]:
neighbor_dict = {}

for cell in daily_h3_risk['h3_cell'].unique():
    neighbors = list(h3.grid_disk(cell, 1))
    neighbors.remove(cell)  # remove self
    neighbor_dict[cell] = neighbors[:4]  # keep 4 neighbors

In [276]:
risk_lookup = daily_h3_risk[['h3_cell', 'date', 'total_risk']]

In [277]:
# Expand h3 -> neighbors
neighbor_df = pd.DataFrame(
    [
        (cell, nbr)
        for cell, nbrs in neighbor_dict.items()
        for nbr in nbrs
    ],
    columns=['h3_cell', 'neighbor_cell']
)

# Attach neighbor risk on same date
neighbor_risk = (
    daily_h3_risk[['h3_cell', 'date']]
    .merge(neighbor_df, on='h3_cell', how='left')
    .merge(
        risk_lookup.rename(
            columns={
                'h3_cell': 'neighbor_cell',
                'total_risk': 'neighbor_risk'
            }
        ),
        on=['neighbor_cell', 'date'],
        how='left'
    )
)

avg_neighbor_risk = (
    neighbor_risk
    .groupby(['h3_cell', 'date'])['neighbor_risk']
    .mean()
    .reset_index(name='avg_neighbor_risk')
)

daily_h3_risk = daily_h3_risk.merge(
    avg_neighbor_risk,
    on=['h3_cell', 'date'],
    how='left'
)

In [278]:
for lag in [1, 2, 3, 7]:
    daily_h3_risk[f'avg_neighbor_risk_lag_{lag}'] = (
        daily_h3_risk
        .groupby('h3_cell')['avg_neighbor_risk']
        .shift(lag)
    )

In [279]:
nbr_grp = daily_h3_risk.groupby('h3_cell')['avg_neighbor_risk']

daily_h3_risk['nbr_roll_mean_7'] = (
    nbr_grp.shift(1)
           .rolling(7, min_periods=1)
           .mean()
           .reset_index(level=0, drop=True)
)

daily_h3_risk['nbr_roll_std_7'] = (
    nbr_grp.shift(1)
           .rolling(7, min_periods=1)
           .std()
           .reset_index(level=0, drop=True)
)

In [280]:
daily_h3_risk.head()

,h3_cell,date,total_risk,total_chalans,avg_neighbor_risk,avg_neighbor_risk_lag_1,avg_neighbor_risk_lag_2,avg_neighbor_risk_lag_3,avg_neighbor_risk_lag_7,nbr_roll_mean_7,nbr_roll_std_7
0,8860144a2dfffff,2023-11-29,48,4,30.0,NaN,NaN,NaN,NaN,NaN,NaN
1,8860144b4dfffff,2023-11-15,57,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8860144b5bfffff,2023-11-29,30,3,48.0,NaN,NaN,NaN,NaN,NaN,NaN
3,8860144b5bfffff,2023-11-30,12,1,NaN,48.0,NaN,NaN,NaN,48.0,NaN
4,8860144b5bfffff,2023-12-01,15,1,NaN,NaN,48.0,NaN,NaN,48.0,NaN


## 5. Creating hexagon's current lag features

In [281]:
daily_h3_risk = daily_h3_risk.sort_values(
    ['h3_cell', 'date']
).reset_index(drop=True)

for lag in [1, 2, 3, 7]:
    daily_h3_risk[f'total_risk_lag{lag}'] = (
        daily_h3_risk
        .groupby('h3_cell')['total_risk']
        .shift(lag)
    )

# Remove first 6 rows of each h3_cell (rows with no lag7)

In [282]:
valid_h3 = (
    daily_h3_risk.groupby('h3_cell')['date']
      .nunique()
      .index
)

df_filtered = daily_h3_risk[daily_h3_risk['h3_cell'].isin(valid_h3)]

In [283]:
df_filtered.columns

Index(['h3_cell', 'date', 'total_risk', 'total_chalans', 'avg_neighbor_risk',
       'avg_neighbor_risk_lag_1', 'avg_neighbor_risk_lag_2',
       'avg_neighbor_risk_lag_3', 'avg_neighbor_risk_lag_7', 'nbr_roll_mean_7',
       'nbr_roll_std_7', 'total_risk_lag1', 'total_risk_lag2',
       'total_risk_lag3', 'total_risk_lag7'],
      dtype='object')

In [284]:
lag_cols = [c for c in df_filtered.columns if 'lag' in c]

df_filtered[lag_cols] = df_filtered[lag_cols].fillna(-999)

In [285]:
df_filtered['date'] = pd.to_datetime(df_filtered['date'])

In [286]:
df_filtered['dow'] = df_filtered['date'].dt.dayofweek

## 6. Modelling

In [287]:
df_filtered = (
    df_filtered
    .sort_values(['h3_cell', 'date'])
    .reset_index(drop=True)
)

FEATURES = [
    'avg_neighbor_risk_lag_1',
    'avg_neighbor_risk_lag_2',
    'avg_neighbor_risk_lag_3',
    'avg_neighbor_risk_lag_7',
    'total_risk_lag1',
    'total_risk_lag2',
    'total_risk_lag3',
    'total_risk_lag7',
    'nbr_roll_std_7',
    'dow'
]

TARGET = 'total_risk'

split_date = df_filtered['date'].quantile(0.8)

train_mask = df_filtered['date'] <= split_date

X_train = df_filtered.loc[train_mask, FEATURES]
y_train = df_filtered.loc[train_mask, TARGET]

X_test = df_filtered.loc[~train_mask, FEATURES]
y_test = df_filtered.loc[~train_mask, TARGET]

print(f"Train rows: {len(X_train):,} (up to {split_date.date()})")
print(f"Test rows : {len(X_test):,} (after {split_date.date()})")

Train rows: 11,593 (up to 2024-01-21)
Test rows : 2,810 (after 2024-01-21)


In [288]:
print(df_filtered['date'].min())
print(split_date)
print(df_filtered['date'].max())

2023-11-10 00:00:00
2024-01-21 00:00:00
2024-03-29 00:00:00


In [246]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

# ── Train XGBoost regressor ──────────────────────────────────
model = xgb.XGBRegressor(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
pred = np.clip(pred, 0, None)  # violation counts can't be negative

mae      = mean_absolute_error(y_test, pred)
r2       = r2_score(y_test, pred)
baseline_mae = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))

print("="*55)
print("NEXT-DAY VIOLATION VOLUME FORECAST — RESULTS")
print("="*55)
print(f"Model MAE        : {mae:.2f} violations/day")
print(f"Naive baseline MAE: {baseline_mae:.2f} violations/day  (always predict the mean)")
print(f"Improvement       : {(1 - mae/baseline_mae)*100:.1f}% better than baseline")
print(f"R² score          : {r2:.3f}")
print(f"EVS  : {evs:.3f}")

NEXT-DAY VIOLATION VOLUME FORECAST — RESULTS
Model MAE        : 10.45 violations/day
Naive baseline MAE: 13.68 violations/day  (always predict the mean)
Improvement       : 23.6% better than baseline
R² score          : 0.283
EVS  : 0.296


In [247]:
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    explained_variance_score
)
import lightgbm as lgb
import numpy as np

# ── Train LightGBM regressor ────────────────────────────────
model = lgb.LGBMRegressor(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
pred = np.clip(pred, 0, None)  # risk can't be negative

# ── Metrics ────────────────────────────────────────────────
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
evs = explained_variance_score(y_test, pred)

baseline_pred = np.repeat(y_train.mean(), len(y_test))
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("=" * 55)
print("NEXT-DAY VIOLATION VOLUME FORECAST — LIGHTGBM")
print("=" * 55)
print(f"Model MAE         : {mae:.2f} violations/day")
print(f"Naive baseline MAE: {baseline_mae:.2f} violations/day")
print(f"Improvement       : {(1 - mae / baseline_mae) * 100:.1f}% better than baseline")
print(f"R² score          : {r2:.3f}")
print(f"EVS               : {evs:.3f}")

NEXT-DAY VIOLATION VOLUME FORECAST — LIGHTGBM
Model MAE         : 10.52 violations/day
Naive baseline MAE: 13.68 violations/day
Improvement       : 23.1% better than baseline
R² score          : 0.260
EVS               : 0.274


In [251]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    explained_variance_score
)
import numpy as np

# ── Train Random Forest regressor ───────────────────────────
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

pred = rf_model.predict(X_test)
pred = np.clip(pred, 0, None)

# ── Metrics ─────────────────────────────────────────────────
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
evs = explained_variance_score(y_test, pred)

baseline_pred = np.repeat(y_train.mean(), len(y_test))
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("=" * 55)
print("NEXT-DAY VIOLATION VOLUME FORECAST — RANDOM FOREST")
print("=" * 55)
print(f"Model MAE         : {mae:.2f} violations/day")
print(f"Naive baseline MAE: {baseline_mae:.2f} violations/day")
print(f"Improvement       : {(1 - mae / baseline_mae) * 100:.1f}% better than baseline")
print(f"R² score          : {r2:.3f}")
print(f"EVS               : {evs:.3f}")

NEXT-DAY VIOLATION VOLUME FORECAST — RANDOM FOREST
Model MAE         : 10.46 violations/day
Naive baseline MAE: 13.68 violations/day
Improvement       : 23.6% better than baseline
R² score          : 0.294
EVS               : 0.308


In [259]:
from sklearn.ensemble import ExtraTreesRegressor

ext_model = ExtraTreesRegressor(
    n_estimators=500,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42
)

ext_model.fit(X_train, y_train)

pred = ext_model.predict(X_test)
pred = np.clip(pred, 0, None)  # risk can't be negative

# ── Metrics ────────────────────────────────────────────────
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
evs = explained_variance_score(y_test, pred)

baseline_pred = np.repeat(y_train.mean(), len(y_test))
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("=" * 55)
print("NEXT-DAY VIOLATION VOLUME FORECAST — CATBOOST")
print("=" * 55)
print(f"Model MAE         : {mae:.2f} violations/day")
print(f"Naive baseline MAE: {baseline_mae:.2f} violations/day")
print(f"Improvement       : {(1 - mae / baseline_mae) * 100:.1f}% better than baseline")
print(f"R² score          : {r2:.3f}")
print(f"EVS               : {evs:.3f}")

NEXT-DAY VIOLATION VOLUME FORECAST — CATBOOST
Model MAE         : 10.40 violations/day
Naive baseline MAE: 13.68 violations/day
Improvement       : 24.0% better than baseline
R² score          : 0.307
EVS               : 0.319


In [252]:
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    explained_variance_score
)
from catboost import CatBoostRegressor
import numpy as np

# ── Train CatBoost regressor ────────────────────────────────
cat_model = CatBoostRegressor(
    iterations=150,
    depth=4,
    learning_rate=0.05,
    random_seed=42,
    verbose=0
)

cat_model.fit(X_train, y_train)

pred = cat_model.predict(X_test)
pred = np.clip(pred, 0, None)  # risk can't be negative

# ── Metrics ────────────────────────────────────────────────
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
evs = explained_variance_score(y_test, pred)

baseline_pred = np.repeat(y_train.mean(), len(y_test))
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("=" * 55)
print("NEXT-DAY VIOLATION VOLUME FORECAST — CATBOOST")
print("=" * 55)
print(f"Model MAE         : {mae:.2f} violations/day")
print(f"Naive baseline MAE: {baseline_mae:.2f} violations/day")
print(f"Improvement       : {(1 - mae / baseline_mae) * 100:.1f}% better than baseline")
print(f"R² score          : {r2:.3f}")
print(f"EVS               : {evs:.3f}")

NEXT-DAY VIOLATION VOLUME FORECAST — CATBOOST
Model MAE         : 10.37 violations/day
Naive baseline MAE: 13.68 violations/day
Improvement       : 24.2% better than baseline
R² score          : 0.331
EVS               : 0.344
